# 2 · Data acquisition & technical indicators
OHLCV for the tickers in `config.py`, the S&P 500 (`^GSPC`) as market index and the 13-week T-bill (`^IRX`) as risk-free rate.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config
from src import data_loader as dl
OFFLINE = False   # True = synthetic sample data (no internet needed)

In [ ]:
if OFFLINE:
    data = dl.generate_sample_data(config.TICKERS, config.BENCHMARK)
    rf = config.DEFAULT_RISK_FREE
else:
    data = dl.fetch_prices(config.TICKERS + [config.BENCHMARK], config.START_DATE, config.END_DATE)
    rf = dl.fetch_risk_free(config.START_DATE, config.END_DATE)
prices = dl.close_matrix(data, config.TICKERS)
print(f'risk-free rate: {rf:.2%}'); prices.tail()

In [ ]:
dl.save_csv(data, '../data/historical_prices.csv')   # long format: Date,Ticker,Open,High,Low,Close,Volume

In [ ]:
from src.indicators import add_indicators
aapl = add_indicators(data['AAPL'])
aapl[['Close','SMA20','EMA12','MACD','RSI14','Stoch_K','BB_Upper','BB_Lower','ATR14']].tail()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
last = aapl.iloc[-250:]
ax[0].plot(last['Close'], label='Close'); ax[0].plot(last['SMA20'], label='SMA20'); ax[0].plot(last['SMA50'], label='SMA50')
ax[0].fill_between(last.index, last['BB_Lower'], last['BB_Upper'], alpha=.15, label='Bollinger'); ax[0].legend()
ax[1].plot(last['MACD'], label='MACD'); ax[1].plot(last['MACD_Signal'], label='Signal'); ax[1].legend()
ax[2].plot(last['RSI14'], label='RSI14'); ax[2].axhline(70, ls=':'); ax[2].axhline(30, ls=':'); ax[2].legend()
plt.tight_layout()